In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
dbutils.widgets.removeAll()

In [0]:
## PARAMETRIZAR CATALOGO A PROD
dbutils.widgets.text("PRM_catalogo","catalogo_desa_intEcommerce")

PRM_catalogo = dbutils.widgets.get("PRM_catalogo")

In [0]:

def read_tablas_Silver():

    df_cliente = spark.table(f"{PRM_catalogo}.silver.Tabla_Cliente") \
        .select(
            col("ID_Cliente").alias("ID_Cliente"), 
            concat(col("Nombre"), lit(" "), col("Apellido")).alias("Nombre_completo"),
            col("Numero_Celular").alias("Numero_Celular")          
        )

    df_tablaproducto = spark.table(f"{PRM_catalogo}.silver.Tabla_Producto") \
        .select(
            col("Nombre_producto").alias("Nombre_producto"),
            col("Cod_producto").alias("Cod_producto"),  
            col("Categoria").alias("Categoria"), 
            col("Precio").alias("Precio")
        ) 

    df_TablaEcommerce = spark.table(f"{PRM_catalogo}.silver.Tabla_IntEcommerce") \
        .where(
                (col("Cod_Tipo_Interaccion") == "TipInt004") & 
                (
                    (col("Cantidad_Producto") != '-') | 
                    (col("Cantidad_Producto").isNotNull()) | 
                    (col("Cantidad_Producto") != ' ')
                )
             )\
        .select(
            trim(col("ID_interaccion")).alias("ID_interaccion"),
            date_format(to_timestamp(col("Fecha_Interaccion"), "M/d/yyyy H:mm"), "MM-yyyy").alias("Periodo_Mes"), 
            trim(col("ID_Cliente")).alias("ID_Cliente"),  
            trim(col("Cod_producto")).alias("Cod_producto"),
            trim(col("Cod_Tipo_Interaccion")).alias("Cod_Tipo_Interaccion"),
            trim(col("Evento")).alias("Evento"),      
            trim(col("Locacion")).alias("Locacion"), 
            col("Cantidad_Producto").cast("bigint").alias("Cantidad_Producto"),
            col("Puntuacion_Producto").alias("Puntuacion_Producto")         
        ) \
             

    return df_cliente, df_tablaproducto, df_TablaEcommerce



In [0]:
def Tablajoin(df_cliente, df_tablaproducto, df_TablaEcommerce):

    df_ClientesCompras = \
        df_TablaEcommerce.alias("A")\
        .join(df_cliente.alias("C"), col("A.ID_Cliente") == col("C.ID_Cliente"), "left")\
        .join(df_tablaproducto.alias("P"), col("A.Cod_producto") == col("P.Cod_producto"), "left")\
            .select(
                col("A.ID_Cliente").alias("ID_Cliente"),	
                col("C.Nombre_completo").alias("Nombre_completo"),
                col("C.Numero_Celular").alias("Numero_Celular"),
                col("P.Nombre_producto").alias("Nombre_producto"),
                col("P.Categoria").alias("Categoria"),       
 ##               date_format(to_date(col("A.Periodo_Mes"), "MM-yyyy"), "MMMM-yyyy").alias("Periodo_Mes"),
                col("A.Cantidad_Producto").alias("Cantidad_Producto"),
                (col("A.Cantidad_Producto") * col("P.Precio")).cast(DecimalType(10,2)).alias("Precio_Compra")   
            ).where(col("C.Nombre_completo").isNotNull())

    return  df_ClientesCompras


In [0]:
def Tabla_Clientes_KPI_Compras(df_ClientesCompras):
    df_prefinal = df_ClientesCompras \
        .groupBy(
            col("ID_Cliente"),
            col("Nombre_completo"),
            col("Numero_Celular")         
        ).agg(
            sum("Precio_Compra").alias("Precio_Compra_Total")
        )

    df_final = df_prefinal.withColumn("KPI_Tipo_Cliente",
            when(col("Precio_Compra_Total") > 17500, "SOCIO_ELITE")
            .when(col("Precio_Compra_Total") > 10500, "SOCIO_PREMIUM")
            .when(col("Precio_Compra_Total") > 5000, "SOCIO_ACTIVO")
            .otherwise("CLIENTE_ESTANDAR")
        ).orderBy(col("Precio_Compra_Total").desc())
    
    return df_final


In [0]:
def main():

    df_cliente,df_tablaproducto,df_TablaEcommerce = read_tablas_Silver()
    
    df_ClientesCompras = Tablajoin(df_cliente, df_tablaproducto, df_TablaEcommerce)
    
    df_final = Tabla_Clientes_KPI_Compras(df_ClientesCompras)
    
    df_final.write.mode("overwrite").saveAsTable(f"{PRM_catalogo}.golden.Clientes_Top_Compras")

main()
